# 03 — Feature Engineering

Demonstrates the Stage 6 feature pipeline (`src/features/`) and, live, the leakage-safety guarantee that `tests/test_features.py` enforces automatically on every run.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
from src.config import RAW_DIR
from src.features.market_features import build_market_features

df = pd.read_parquet(RAW_DIR / 'market_prices' / 'NVDA.parquet')
features = build_market_features(df)
features[['date', 'close', 'return_1d', 'rsi_14d', 'momentum_10d', 'rolling_vol_30d']].tail(10)

,date,close,return_1d,rsi_14d,momentum_10d,rolling_vol_30d
814,2026-08-31,220.779999,0.014847,52.540272,0.019124,0.028247
815,2026-09-01,217.440002,-0.015128,44.576746,0.004773,0.028249
816,2026-09-02,224.410004,0.032055,49.336514,0.033709,0.028520
817,2026-09-03,228.449997,0.018003,52.317877,0.046726,0.028465
818,2026-09-04,230.360001,0.008361,53.677992,0.047967,0.028385
819,2026-09-08,225.729996,-0.020099,54.154522,0.018908,0.026845
820,2026-09-09,223.669998,-0.009126,54.244825,0.004793,0.026961
821,2026-09-10,218.360001,-0.023740,50.986022,-0.022880,0.026419
822,2026-09-11,218.289993,-0.000321,52.395646,-0.018939,0.026117
823,2026-09-14,210.960007,-0.033579,51.640219,-0.049066,0.026558


## Live leakage-safety demonstration

Mutating every row after a cutoff must not change any feature value before it — this is the same check `tests/test_features.py::TestLeakageSafety` runs automatically; here it's shown directly against real NVDA data.

In [2]:
cutoff = 200
mutated = df.copy()
mutated.loc[cutoff + 1:, 'close'] *= 5.0
features_mutated = build_market_features(mutated)

# NaN-aware comparison: a plain `==` would flag the RSI warm-up window's
# NaN values as "different" from each other (NaN != NaN is always True in
# pandas), which would misreport a false leak. pandas.Series.equals treats
# NaN as equal to NaN, matching what tests/test_features.py actually checks.
before = features['rsi_14d'].iloc[:cutoff].reset_index(drop=True)
after = features_mutated['rsi_14d'].iloc[:cutoff].reset_index(drop=True)
unchanged = before.equals(after)
print(f'All pre-cutoff rsi_14d values unchanged after mutating future data: {unchanged}')

All pre-cutoff rsi_14d values unchanged after mutating future data: True


Full feature dictionary with every column's formula and window: see `FEATURES.md`.